# DefectVision AI - Building/Structural Defect Detection
## Phase 1: Train YOLOv8 on Structural Damage Dataset

This notebook trains a YOLOv8 model to detect structural defects in buildings:
- Cracks
- Spalling
- Corrosion
- Exposed rebar

**Works on: Kaggle | Google Colab | Local (GPU or CPU)**

The notebook auto-detects your environment and adjusts batch size, device, and output paths accordingly.

| Environment | GPU | Batch Size | Notes |
|---|---|---|---|
| **Kaggle** | P100 (16GB) or T4 (15GB) | 16 | Enable GPU in Settings > Accelerator |
| **Google Colab** | T4 (15GB) | 16 | Runtime > Change runtime type > GPU |
| **Local (GTX 1650 Ti)** | 1650 Ti (4GB) | 8 | Install CUDA + PyTorch with GPU support |
| **Local (CPU)** | None | 4 | Very slow, not recommended |

## Step 1: Detect Environment & Install Dependencies

In [ ]:
import os, sys

# --- Auto-detect environment ---
def detect_environment():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    try:
        import google.colab
        return "colab"
    except ImportError:
        pass
    return "local"

ENV = detect_environment()
print(f"Detected environment: {ENV.upper()}")

# --- Install dependencies ---
if ENV in ("kaggle", "colab"):
    os.system("pip install ultralytics roboflow -q")
    print("Installed ultralytics + roboflow")
else:
    # Local: assume deps are already installed via requirements.txt
    # If not, uncomment the line below:
    # os.system(f"{sys.executable} -m pip install ultralytics roboflow -q")
    print("Local mode: using existing packages (run 'pip install ultralytics roboflow' if missing)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 93.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.38.0, but you have google-auth 2.47.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyt

## Step 2: Check GPU & Configure Training Parameters

In [2]:
import torch

print(f"Environment:     {ENV.upper()}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

# --- Auto-configure based on GPU ---
if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    VRAM_GB = getattr(props, 'total_memory', getattr(props, 'total_mem', 0)) / 1e9
    DEVICE = 0
    print(f"GPU:             {GPU_NAME}")
    print(f"VRAM:            {VRAM_GB:.1f} GB")
    print(f"CUDA version:    {torch.version.cuda}")

    # Auto-select batch size based on VRAM
    if VRAM_GB >= 12:
        BATCH_SIZE = 16
        MODEL_SIZE = "yolov8s.pt"   # small (11.2M params) - more accurate
    elif VRAM_GB >= 6:
        BATCH_SIZE = 12
        MODEL_SIZE = "yolov8s.pt"
    elif VRAM_GB >= 4:
        BATCH_SIZE = 8
        MODEL_SIZE = "yolov8n.pt"   # nano (3.2M params) - fits 4GB
    else:
        BATCH_SIZE = 4
        MODEL_SIZE = "yolov8n.pt"
else:
    GPU_NAME = "CPU"
    VRAM_GB = 0
    DEVICE = "cpu"
    BATCH_SIZE = 4
    MODEL_SIZE = "yolov8n.pt"
    print("WARNING: No CUDA GPU detected. Training on CPU will be very slow!")
    if ENV == "kaggle":
        print("  -> Go to Settings (right panel) > Accelerator > GPU P100 or T4x2")
    elif ENV == "colab":
        print("  -> Go to Runtime > Change runtime type > GPU (T4)")

# ========================================================================
# OVERRIDE: Uncomment and change these if you want to force specific values
# ========================================================================
# BATCH_SIZE = 8
# MODEL_SIZE = "yolov8s.pt"   # options: yolov8n.pt, yolov8s.pt, yolov8m.pt
# DEVICE = 0                  # 0 = first GPU, "cpu" = CPU

EPOCHS = 100
PATIENCE = 20
IMG_SIZE = 640

print(f"\n--- Training Config ---")
print(f"Model:      {MODEL_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs:     {EPOCHS} (early stop patience={PATIENCE})")
print(f"Image size: {IMG_SIZE}")
print(f"Device:     {DEVICE}")

PyTorch version: 2.8.0+cu126
CUDA available: False


## Step 3: Download Structural Damage Dataset from Roboflow

This dataset contains images of buildings/structures with annotated damage.

**To get your free Roboflow API key:**
1. Go to [roboflow.com](https://roboflow.com) and create a free account
2. Go to Settings > API Keys
3. Copy your API key

**For Kaggle:** You can add it as a Kaggle Secret named `ROBOFLOW_API_KEY` (Add-ons > Secrets), or paste it directly below.

**For Colab / Local:** Paste it in the cell below.

In [ ]:
from roboflow import Roboflow

# -------------------------------------------------------------------
# PASTE YOUR ROBOFLOW API KEY HERE (or use Kaggle Secrets)
# -------------------------------------------------------------------
ROBOFLOW_API_KEY = ""  # <-- paste your key inside the quotes
# -------------------------------------------------------------------

# Try to load from Kaggle Secrets if not set
if not ROBOFLOW_API_KEY and ENV == "kaggle":
    try:
        from kaggle_secrets import UserSecretsClient
        ROBOFLOW_API_KEY = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
        print("Loaded API key from Kaggle Secrets")
    except Exception:
        pass

# Try environment variable as fallback
if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "")

if not ROBOFLOW_API_KEY:
    raise ValueError(
        "No Roboflow API key found!\n"
        "  Option 1: Paste it directly in ROBOFLOW_API_KEY above\n"
        "  Option 2: (Kaggle) Add secret named ROBOFLOW_API_KEY\n"
        "  Option 3: Set env var ROBOFLOW_API_KEY"
    )

# Download dataset
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("university-bswxt").project("crack-bphdr")
version = project.version(2)

# Choose download location based on environment
if ENV == "kaggle":
    download_dir = "/kaggle/working/datasets/building"
elif ENV == "colab":
    download_dir = "/content/datasets/building"
else:
    download_dir = "datasets/building"

dataset = version.download("yolov8", location=download_dir)
print(f"\nDataset downloaded to: {dataset.location}")

upload and label your dataset, and get an API KEY here: https://app.roboflow.com/?model=undefined&ref=undefined
loading Roboflow workspace...


RoboflowError: {"error":{"message":"This API key does not exist (or has been revoked).","status":401,"type":"OAuthException","hint":"You may retrieve your API key via the Roboflow Dashboard. Go to Account > Roboflow Keys to retrieve yours.","key":"YOUR_API_KEY_HERE"}}

## Step 4: Explore the Dataset

In [ ]:
import yaml
from pathlib import Path

# Read the data.yaml to see class info
data_yaml_path = os.path.join(dataset.location, "data.yaml")
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print("Dataset configuration:")
print(f"  Classes: {data_config.get('names', 'N/A')}")
print(f"  Number of classes: {data_config.get('nc', 'N/A')}")
print(f"  data.yaml path: {data_yaml_path}")

# Count images in each split
total_images = 0
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(dataset.location, split, 'images')
    if os.path.exists(img_dir):
        count = len(os.listdir(img_dir))
        total_images += count
        print(f"  {split}: {count} images")
    else:
        print(f"  {split}: directory not found")
print(f"  TOTAL: {total_images} images")

In [ ]:
# Visualize some sample images
import matplotlib.pyplot as plt
import cv2
import random

train_img_dir = os.path.join(dataset.location, 'train', 'images')
sample_images = random.sample(os.listdir(train_img_dir), min(8, len(os.listdir(train_img_dir))))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, img_name in zip(axes.flatten(), sample_images):
    img = cv2.imread(os.path.join(train_img_dir, img_name))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(img_name[:20], fontsize=8)
    ax.axis('off')
plt.suptitle('Sample Training Images - Building/Structural Defects', fontsize=14)
plt.tight_layout()
plt.show()

## Step 5: Train YOLOv8 Model

Training config was auto-detected in Step 2. Estimated times:
- **Kaggle P100 / Colab T4** (yolov8s, batch=16): ~25-35 min
- **Local GTX 1650 Ti** (yolov8n, batch=8): ~30-45 min
- **CPU**: ~3-6 hours (not recommended)

In [ ]:
from ultralytics import YOLO

print(f"Loading pretrained {MODEL_SIZE}...")
model = YOLO(MODEL_SIZE)

# Train on building/structural defect dataset
results = model.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    name="building_defect_detector",
    patience=PATIENCE,
    save=True,
    plots=True,
    device=DEVICE,
    workers=2 if ENV == "local" else 4,
    exist_ok=True,
    # augmentation for better crack detection
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    flipud=0.5,
    mosaic=1.0,
    scale=0.5,
)

print("\nTraining complete!")

## Step 6: Evaluate the Model

In [ ]:
# Validate on the validation set
metrics = model.val()

print(f"\n=== Building Defect Detection Results ===")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

In [ ]:
# Show training curves
from IPython.display import Image, display

results_dir = Path("runs/detect/building_defect_detector")

for plot_name in ["results.png", "confusion_matrix.png", "val_batch0_pred.png"]:
    plot_path = results_dir / plot_name
    if plot_path.exists():
        print(f"\n--- {plot_name} ---")
        display(Image(filename=str(plot_path), width=800))

## Step 7: Test on Sample Images

In [ ]:
# Run inference on validation images (lower conf for cracks)
INFERENCE_CONF = 0.15

val_img_dir = os.path.join(dataset.location, 'valid', 'images')
if not os.path.exists(val_img_dir):
    val_img_dir = os.path.join(dataset.location, 'test', 'images')

val_images = os.listdir(val_img_dir)
sample_val = random.sample(val_images, min(6, len(val_images)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, img_name in zip(axes.flatten(), sample_val):
    img_path = os.path.join(val_img_dir, img_name)
    preds = model(img_path, conf=INFERENCE_CONF, verbose=False)
    annotated = preds[0].plot()
    annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    ax.imshow(annotated)
    ax.set_title(img_name[:25], fontsize=9)
    ax.axis('off')

plt.suptitle(f'YOLOv8 Predictions (conf>={INFERENCE_CONF}) - Building Defects', fontsize=14)
plt.tight_layout()
plt.show()

## Step 8: Save / Download Trained Weights

- **Local**: Weights are auto-copied to `models/building_yolo_best.pt`
- **Kaggle**: Saved to `/kaggle/working/` (appears in Output tab)
- **Colab**: Auto-downloads to your browser

In [ ]:
import shutil

# Locate best weights
best_weights = results_dir / "weights" / "best.pt"
if not best_weights.exists():
    # Search for it
    for p in Path("runs/detect").rglob("best.pt"):
        best_weights = p
        break

if not best_weights.exists():
    print("ERROR: Could not find best.pt anywhere in runs/detect/")
else:
    file_size = best_weights.stat().st_size / 1e6
    print(f"Found best weights: {best_weights} ({file_size:.1f} MB)")

    if ENV == "local":
        # Copy directly to project models/ folder
        models_dir = Path("../models") if Path("../models").parent.exists() else Path("models")
        models_dir.mkdir(exist_ok=True)
        output_path = models_dir / "building_yolo_best.pt"
        shutil.copy2(best_weights, output_path)
        print(f"Copied to: {output_path.resolve()}")
        print("Ready to use with: python app.py")

    elif ENV == "kaggle":
        # Copy to /kaggle/working/ so it appears in Output tab
        output_path = Path("/kaggle/working/building_yolo_best.pt")
        shutil.copy2(best_weights, output_path)
        print(f"Saved to: {output_path}")
        print("Download from the 'Output' tab on the right panel")
        print("Then place it in your local project at: models/building_yolo_best.pt")

    elif ENV == "colab":
        # Auto-download via Colab
        shutil.copy2(best_weights, "building_yolo_best.pt")
        try:
            from google.colab import files
            files.download("building_yolo_best.pt")
            print("Download started! Save it to: models/building_yolo_best.pt")
        except Exception as e:
            print(f"Auto-download failed: {e}")
            print("Manually download building_yolo_best.pt from the Files panel")